# Movie Recommender Evaluation

This notebook evaluates a content-based movie recommendation model and generates visuals for a portfolio.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from recommender import MovieRecommender

sns.set_theme(style='whitegrid')

In [ ]:
data_path = ROOT / 'data' / 'movies_sample.csv'
movies_df = pd.read_csv(data_path)

recommender = MovieRecommender()
recommender.fit(movies_df)

print(f'Dataset size: {len(movies_df)} movies')
movies_df.head(3)

In [ ]:
seed_title = 'Inception'
top_n = 10
results = recommender.recommend_similar(seed_title, top_n=top_n)

recommendations_df = pd.DataFrame([
    {'title': r.title, 'year': r.year, 'score': r.score} for r in results
])
recommendations_df

In [ ]:
def genre_tokens(raw_genres: str) -> set[str]:
    if not isinstance(raw_genres, str):
        return set()
    return {token.strip().lower() for token in raw_genres.split(' ') if token.strip()}

def evaluate_recommendations(df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    temp = df.copy()
    temp['title_norm'] = temp['title'].astype(str).str.strip().str.lower()
    title_to_genres = {
        row['title_norm']: genre_tokens(row.get('genres', ''))
        for _, row in temp.iterrows()
    }

    overlap_rates = []
    avg_scores = []
    unique_recommended = set()

    for title in temp['title'].head(min(len(temp), 25)):
        recs = recommender.recommend_similar(title, top_n=k)
        rec_titles = [r.title.strip().lower() for r in recs]
        rec_scores = [r.score for r in recs]

        unique_recommended.update(rec_titles)
        avg_scores.append(float(np.mean(rec_scores)))

        source_genres = title_to_genres.get(title.strip().lower(), set())
        hits = 0
        for rec_title in rec_titles:
            if source_genres.intersection(title_to_genres.get(rec_title, set())):
                hits += 1
        overlap_rates.append(hits / k if k else 0.0)

    coverage = len(unique_recommended) / max(len(temp), 1)

    return pd.DataFrame([
        {'metric': 'Genre Overlap@K', 'value': float(np.mean(overlap_rates))},
        {'metric': 'Average Similarity Score', 'value': float(np.mean(avg_scores))},
        {'metric': 'Catalog Coverage', 'value': float(coverage)},
    ])

metrics_df = evaluate_recommendations(movies_df, k=5)
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=metrics_df, x='metric', y='value', ax=axes[0], palette='YlOrBr')
axes[0].set_title('Evaluation Metrics')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)

sns.histplot(recommendations_df['score'], bins=8, kde=True, ax=axes[1], color='#b45309')
axes[1].set_title('Similarity Score Distribution (Inception Top-10)')
axes[1].set_xlabel('Cosine Similarity')

plt.tight_layout()
plt.show()

## Notes
- This evaluation uses a small sample dataset; use a larger metadata dataset for stronger metrics.
- You can reuse this notebook with your own CSV by changing `data_path` above.